In [ ]:
!pip install -q langgraph groq "langchain>=0.1.0" langchain_core langchain_groq PyPDF2 python-docx langchain_community mammoth email-validator


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/

In [ ]:

import os
import json
import datetime
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import io
from typing import Dict, List, Annotated, TypedDict, Union, Tuple

import PyPDF2
import docx
import mammoth
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import END, StateGraph

In [ ]:

GROQ_API_KEY = ""

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=GROQ_API_KEY)

class HiringState(TypedDict):
    """Represents the state of our hiring workflow."""
    candidate: Dict
    screening_result: Dict
    emails_sent: List[Dict]
    final_decision: str
    next_step: str
    history: List

THRESHOLD_SCORE = 75

def extract_text_from_pdf(file_content):
    """Extract text from PDF file content."""
    pdf_reader = PyPDF2.PdfReader(io.BytesIO(file_content))
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() + "\n"
    return text

def extract_text_from_docx(file_content):
    """Extract text from DOCX file content."""
    doc = docx.Document(io.BytesIO(file_content))
    text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
    return text

def extract_text_from_doc(file_content):
    """Extract text from DOC file content using mammoth."""
    result = mammoth.convert_to_html(io.BytesIO(file_content))
    text = result.value
    return text

def extract_resume_text(file_content, file_type):
    """Extract text from resume file."""
    if file_type.lower().endswith('.pdf'):
        return extract_text_from_pdf(file_content)
    elif file_type.lower().endswith('.docx'):
        return extract_text_from_docx(file_content)
    elif file_type.lower().endswith('.doc'):
        return extract_text_from_doc(file_content)
    else:
        return "Unsupported file format"

In [ ]:


def screen_resume(state: HiringState) -> HiringState:
    """Screen a candidate's resume for job fit."""

    candidate = state["candidate"]
    job = candidate.get("job_description", "")
    resume = candidate.get("resume_text", "")

    prompt = f"""
    You are a Resume Screening Agent. Your task is to evaluate if a candidate is suitable for a job based on their resume.

    JOB DESCRIPTION:
    {job}

    CANDIDATE RESUME:
    {resume}

    Please analyze the resume against the job requirements and provide a detailed assessment.
    Return your analysis as JSON with the following fields:
    - qualified: Boolean (true/false) indicating if the candidate meets basic qualifications
    - overall_score: Integer from 0-100 indicating overall fit
    - strengths: List of the candidate's strengths relevant to the position
    - gaps: List of areas where the candidate falls short
    - recommendation: String with a brief recommendation of why the candidate should or should not proceed
    """

    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)

    try:
        result_text = response.content
        start_idx = result_text.find('{')
        end_idx = result_text.rfind('}') + 1
        if start_idx >= 0 and end_idx > start_idx:
            json_str = result_text[start_idx:end_idx]
            result = json.loads(json_str)
        else:
            raise ValueError("No valid JSON found in response")
    except Exception as e:
        print(f"Error parsing JSON response: {e}")
        result = {
            "qualified": False,
            "overall_score": 0,
            "strengths": [],
            "gaps": ["Failed to properly analyze resume"],
            "recommendation": "Unable to analyze resume properly"
        }

    new_state = state.copy()
    new_state["screening_result"] = result
    new_state["history"].append({"agent": "Resume Screener", "action": "Screened resume", "result": result})

    # Determine next step based on score threshold
    # if result.get("overall_score", 0) >= THRESHOLD_SCORE:
    #     new_state["next_step"] = "schedule_interview"
    #     new_state["final_decision"] = "Proceed to interview"
    # else:
    #     new_state["next_step"] = "send_rejection"
    #     new_state["final_decision"] = "Rejected based on resume screening"
    # In the screen_resume function:
    if result.get("overall_score", 0) >= THRESHOLD_SCORE:

        new_state["next_step"] = "schedule_interview"
        new_state["final_decision"] = "Proceed to interview"
    else:
        new_state["next_step"] = "send_rejection"
        new_state["final_decision"] = "Rejected based on resume screening"

    return new_state

In [ ]:


def schedule_interview(state: HiringState) -> HiringState:
    """Schedule interview for qualified candidates."""

    candidate = state["candidate"]
    candidate_name = candidate.get("name", "Candidate")
    candidate_email = candidate.get("email", "candidate@example.com")

    today = datetime.datetime.now()
    next_day = today + datetime.timedelta(days=1)
    if next_day.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
        days_to_add = 7 - next_day.weekday()
        next_day = today + datetime.timedelta(days=days_to_add)

    interview_date = next_day.strftime("%A, %B %d, %Y")
    interview_time = "10:00 AM"
    interviewer = candidate.get("interviewer", "HR Manager")
    interviewer_email = candidate.get("interviewer_email", "hr@company.com")

    # Generate email to candidate
    candidate_email_content = f"""
    Subject: Interview Invitation - {candidate.get("position", "Position")}

    Dear {candidate_name},

    We're pleased to inform you that your resume has been selected for the next round of our hiring process for the {candidate.get("position", "Position")} role.

    Your interview has been scheduled for:
    Date: {interview_date}
    Time: {interview_time}
    Location: {candidate.get("interview_location", "Online via Zoom")}

    Please confirm your availability for this time by replying to this email.

    We look forward to speaking with you!

    Best regards,
    {candidate.get("company_name", "Company")} Hiring Team
    """

    interviewer_email_content = f"""
    Subject: Interview Scheduled - {candidate_name} for {candidate.get("position", "Position")}

    Dear {interviewer},

    An interview has been scheduled with {candidate_name} for the {candidate.get("position", "Position")} position.

    Interview details:
    Date: {interview_date}
    Time: {interview_time}
    Candidate Email: {candidate_email}

    The candidate's resume and screening results are attached for your review.

    Thank you,
    Automated Hiring System
    """


    emails = [
        {
            "recipient": candidate_email,
            "subject": f"Interview Invitation - {candidate.get('position', 'Position')}",
            "content": candidate_email_content
        },
        {
            "recipient": interviewer_email,
            "subject": f"Interview Scheduled - {candidate_name}",
            "content": interviewer_email_content
        }
    ]

    new_state = state.copy()
    new_state["emails_sent"] = emails
    new_state["history"].append({
        "agent": "Interview Scheduler",
        "action": "Scheduled interview",
        "result": {
            "interview_date": interview_date,
            "interview_time": interview_time
        }
    })
    new_state["next_step"] = "end"

    return new_state

In [ ]:


def send_rejection(state: HiringState) -> HiringState:
    """Send rejection email to candidate."""

    candidate = state["candidate"]
    candidate_name = candidate.get("name", "Candidate")
    candidate_email = candidate.get("email", "candidate@example.com")
    screening_result = state["screening_result"]

    rejection_email_content = f"""
    Subject: Regarding Your Application for {candidate.get("position", "Position")}

    Dear {candidate_name},

    Thank you for your interest in the {candidate.get("position", "Position")} role at {candidate.get("company_name", "Company")}.

    After careful consideration of your application, we regret to inform you that we will not be moving forward with your candidacy at this time.

    We appreciate the time you took to apply and wish you the best in your job search.

    Sincerely,
    {candidate.get("company_name", "Company")} Hiring Team
    """


    emails = [
        {
            "recipient": candidate_email,
            "subject": f"Regarding Your Application for {candidate.get('position', 'Position')}",
            "content": rejection_email_content
        }
    ]

    new_state = state.copy()
    new_state["emails_sent"] = emails
    new_state["history"].append({
        "agent": "Rejection Notifier",
        "action": "Sent rejection email"
    })
    new_state["next_step"] = "end"

    return new_state

In [ ]:


def router(state: HiringState) -> str:
    """Route to the next step based on state."""
    return state["next_step"]

workflow = StateGraph(HiringState)


workflow.add_node("resume_screener", screen_resume)
workflow.add_node("schedule_interview", schedule_interview)
workflow.add_node("send_rejection", send_rejection)


workflow.set_entry_point("resume_screener")

workflow.add_conditional_edges(
    "resume_screener",
    router,
    {
        "schedule_interview": "schedule_interview",
        "send_rejection": "send_rejection"    }
)

workflow.add_edge("schedule_interview", END)
workflow.add_edge("send_rejection", END)

app = workflow.compile()
# Initialize the graph
# workflow = StateGraph(HiringState)

# # Add nodes
# workflow.add_node("resume_screener", screen_resume)
# workflow.add_node("schedule_interview", schedule_interview)
# workflow.add_node("send_rejection", send_rejection)

# # Add edges
# workflow.add_edge("resume_screener", "schedule_interview")
# workflow.add_edge("resume_screener", "send_rejection")
# workflow.add_edge("schedule_interview", END)
# workflow.add_edge("send_rejection", END)

# # Set the entry point
# workflow.set_entry_point("resume_screener")

# # Compile the graph
# app = workflow.compile()

def process_candidate_resume(file_content, file_type, candidate_info):
    """Process a candidate's resume and run it through the hiring workflow."""

    resume_text = extract_resume_text(file_content, file_type)

    candidate_info["resume_text"] = resume_text

    state = {
        "candidate": candidate_info,
        "screening_result": {},
        "emails_sent": [],
        "final_decision": "",
        "next_step": "resume_screener",
        "history": []
    }

    print("🔄 STARTING AUTOMATED HIRING WORKFLOW\n")
    print("=" * 80)

    for output in app.stream(state):
        if "resume_screener" in output:
            result = output["resume_screener"]
            print("\n📄 RESUME SCREENING RESULTS:")
            print(f"Qualified: {result['screening_result'].get('qualified', 'N/A')}")
            print(f"Overall Score: {result['screening_result'].get('overall_score', 'N/A')}/100")

            print("\nStrengths:")
            for strength in result['screening_result'].get('strengths', []):
                print(f"- {strength}")

            print("\nGaps:")
            for gap in result['screening_result'].get('gaps', []):
                print(f"- {gap}")

            print(f"\nRecommendation: {result['screening_result'].get('recommendation', 'N/A')}")
            print("\n" + "=" * 80)

        elif "schedule_interview" in output:
            result = output["schedule_interview"]
            print("\n📅 INTERVIEW SCHEDULED")
            print("Email sent to candidate:")
            print(result['emails_sent'][0]['content'])
            print("\nEmail sent to interviewer:")
            print(result['emails_sent'][1]['content'])
            print("\n" + "=" * 80)

        elif "send_rejection" in output:
            result = output["send_rejection"]
            print("\n❌ REJECTION EMAIL SENT")
            print(result['emails_sent'][0]['content'])
            print("\n" + "=" * 80)

    print("\n🏁 HIRING PROCESS COMPLETED")
    # print(f"Final Decision: {output.get('final_decision', 'No decision recorded')}")
    # print("=" * 80)

    return None

In [ ]:


def upload_and_process_resume():
    """Upload and process a real resume file."""
    from google.colab import files

    print("Please upload a resume file (PDF, DOCX, or DOC):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    file_name = list(uploaded.keys())[0]
    file_content = uploaded[file_name]

    print(f"\nFile '{file_name}' uploaded successfully.")

    print("\nPlease provide candidate information:")
    name = input("Candidate Name: ")
    email = input("Candidate Email: ")
    position = input("Position Applied For: ")

    print("\nPlease provide job description (press Enter twice when done):")
    job_description = input("Enter Job Description")

    # while True:
    #   line = input()
    #   if line.strip().upper() == "END":
    #     break
    #   job_description += line + "\n"

    candidate_info = {
        "name": name,
        "email": email,
        "position": position,
        "company_name": "Your Company",
        "job_description": job_description,
        "interviewer": "HR Manager",
        "interviewer_email": "hr@yourcompany.com",
        "interview_location": "Company HQ, Room 302"
    }

    return process_candidate_resume(file_content, file_name, candidate_info)

print("Testing with sample data...")

upload_and_process_resume()

Testing with sample data...
Please upload a resume file (PDF, DOCX, or DOC):


Saving Muhammad Asad Resume.pdf to Muhammad Asad Resume (7).pdf

File 'Muhammad Asad Resume (7).pdf' uploaded successfully.

Please provide candidate information:
Candidate Name: Asad
Candidate Email: asadrana0086@gmail.com
Position Applied For: AI Engineer

Please provide job description (press Enter twice when done):
Enter Job DescriptionAbout the job About Burq  Burq started with an ambitious mission: how can we turn the complex process of offering delivery into a simple turnkey solution.  We started with building the largest network of delivery networks, partnering with some of the biggest delivery companies. We then made it extremely easy for businesses to plug into our network and start offering delivery to their customers. Now, we're powering deliveries for some of the fastest-growing companies from retailers to startups.  It's a big mission and now we want you to join us to make it even bigger! 🚀  We're already backed by some of the Valley's leading venture capitalists, includi